In [0]:
# Revenue & Payments Q&A Agent — Manual Setup Guide

## Overview

This guide walks you through manually creating a **Genie Agent** on top of the Wanderbricks Semantic Layer.

**What this agent does:** Answers natural-language questions about revenue trends, payment method performance, payment success rates, and transaction-level payment details.

**Architecture:**
```
User Question (natural language)
        ↓
   Genie Agent
        ↓
   SQL Generation (via LLM)
        ↓
   Semantic Layer Views + Gold Tables
        ↓
   SQL Warehouse (read-only query)
        ↓
   Answer (table + visualization)
```

**This is vectorless RAG** — Genie retrieves answers by generating SQL against your semantic layer, not by querying a vector index.

---

## Prerequisites

1. **Semantic layer exists** — `workspace.wanderbricks_semantic` schema with metric views
2. **Gold tables exist** — `workspace.wanderbricks_gold` schema with fact/dim tables
3. **SQL Warehouse running** — Start your Serverless Starter Warehouse (XXSMALL) before testing
4. **Unity Catalog permissions** — You need `USE SCHEMA` on the semantic and gold schemas

---

## What You'll Build (7 Steps)

| Step | What | Time |
|---|---|---|
| 1 | Create the Genie Agent | 2 min |
| 2 | Attach data sources (views + tables) | 5 min |
| 3 | Add instructions (business context) | 5 min |
| 4 | Add example SQL queries (5+ recommended) | 15 min |
| 5 | Configure knowledge store (joins, expressions) | 10 min |
| 6 | Test with benchmark questions | 10 min |
| 7 | Quality tuning (confidence voting, refine) | Ongoing |

---

In [0]:
%sql
-- ============================================================
-- STEP 0: PREREQUISITES CHECK
-- Run this cell to verify your semantic layer is ready
-- ============================================================

-- Check semantic layer views exist
SELECT table_name, table_type, comment 
FROM system.information_schema.tables 
WHERE table_catalog = 'workspace' 
  AND table_schema = 'wanderbricks_semantic'
ORDER BY table_name;

-- Check gold layer tables exist
SELECT table_name, table_type 
FROM system.information_schema.tables 
WHERE table_catalog = 'workspace' 
  AND table_schema = 'wanderbricks_gold'
ORDER BY table_name;

-- Verify mv_revenue_metrics is queryable with MEASURE() syntax
SELECT 
  payment_method,
  MEASURE(total_revenue) AS revenue,
  MEASURE(total_payments) AS payments,
  MEASURE(payment_success_rate) AS success_rate
FROM workspace.wanderbricks_semantic.mv_revenue_metrics
GROUP BY payment_method
ORDER BY revenue DESC;

-- Verify fact_payments has data
SELECT COUNT(*) AS payment_count, 
       MIN(payment_date) AS earliest_payment,
       MAX(payment_date) AS latest_payment
FROM workspace.wanderbricks_gold.fact_payments;

payment_count,earliest_payment,latest_payment
49638,2022-12-21T07:54:22.000Z,2025-07-30T23:59:59.000Z


In [0]:
# Step 1: Create the Genie Agent

## Instructions (follow in Databricks UI)

1. **Navigate to Genie** — In the left sidebar, click **Genie** (or go to **Catalog → Genie**)
2. **Click "Create Agent"** (or "+ New Agent")
3. **Name it:** `Wanderbricks Revenue & Payments Agent`
4. **Description:** `Answers natural-language questions about revenue trends, payment methods, payment success rates, and transaction-level payment details for the Wanderbricks vacation rental platform.`
5. **Click Create**

### What happens next
- The Genie Agent opens in editing mode
- Genie Code may auto-launch and suggest context — **dismiss it for now** (we'll add context manually to learn the process)
- You'll see tabs: **Chat | Configure | Examples | Benchmarks**

### Key concept
> A Genie Agent only queries tables **explicitly attached** under its Sources. It will NOT query tables that aren't attached, even if your instructions reference them. So Step 2 (adding data sources) is critical.

---

In [0]:
# Step 2: Attach Data Sources

## Instructions (follow in Databricks UI)

1. Click the **Configure** tab
2. Click **Data** in the left panel
3. Click **Add** to attach each table/view below

### Tables to attach (in this order)

| # | Fully Qualified Name | Type | Why |
|---|---|---|---|
| 1 | `workspace.wanderbricks_semantic.mv_revenue_metrics` | Metric View | Primary KPI view — revenue, payments, success rate by method/time |
| 2 | `workspace.wanderbricks_semantic.vw_executive_dashboard` | View | Cross-domain executive summary (revenue + bookings + users + hosts) |
| 3 | `workspace.wanderbricks_gold.fact_payments` | View | Transaction-level payment detail for drill-down queries |
| 4 | `workspace.wanderbricks_gold.fact_bookings` | View | Booking context (links payments to bookings) |
| 5 | `workspace.wanderbricks_gold.dim_date` | View | Time dimension for date-based filtering |
| 6 | `workspace.wanderbricks_gold.dim_users` | View | User context (who made the payment) |
| 7 | `workspace.wanderbricks_gold.dim_hosts` | View | Host context (who received the payment) |
| 8 | `workspace.wanderbricks_gold.dim_properties` | View | Property context (what was booked) |

### Important notes

- **Metric views require MEASURE() syntax** — Genie needs example queries to learn this (Step 4)
- **Only attach what's needed** — Too many tables confuse Genie. For this agent, the 8 tables above are the right scope.
- **Verify each table** — After adding, click on each table name to verify columns and sample data look correct

---

In [0]:
# Step 3: Add Instructions

## Instructions (follow in Databricks UI)

1. Click the **Configure** tab
2. Click **Instructions** in the left panel
3. Paste the text below into the **General instructions** box
4. Click **Save**

---

## COPY-PASTE THIS INSTRUCTIONS TEXT:

```
You are a Revenue & Payments analyst for Wanderbricks, a vacation rental platform (like Airbnb).

## Business Context
- Wanderbricks is a vacation rental marketplace where guests book properties hosted by hosts.
- Payments are made by guests for bookings using one of 5 payment methods: credit_card, paypal, apple_pay, google_pay, bank_transfer.
- Each payment has a status: pending, completed, failed, or refunded.
- Only "completed" payments contribute to revenue. "Failed" payments are attempted but unsuccessful.
- Revenue = sum of payment amounts where status = 'completed'.

## Metric View Usage (IMPORTANT)
- The view mv_revenue_metrics is a METRIC VIEW. You MUST use MEASURE() function to query its measures.
- Dimensions (group by columns): payment_year, payment_month, payment_method
- Measures (require MEASURE()): total_revenue, total_payments, avg_payment_amount, successful_payments, failed_payments, payment_success_rate
- Example: SELECT payment_method, MEASURE(total_revenue) FROM workspace.wanderbricks_semantic.mv_revenue_metrics GROUP BY payment_method

## Payment Status Definitions
- successful_payments = count where status = 'completed'
- failed_payments = count where status = 'failed'
- payment_success_rate = successful_payments / total_payments (as a decimal, e.g., 0.91 = 91%)
- "Refunded" and "pending" payments are excluded from both successful and failed counts.

## Date Handling
- payment_month is a TIMESTAMP representing the first day of the month (e.g., 2025-07-01 00:00:00)
- payment_year is an INT (e.g., 2025)
- When users ask for "last 3 months" or "recent", use the latest payment_year and payment_month in the data.

## vw_executive_dashboard
- This is a standard SQL view (NOT a metric view) — no MEASURE() needed.
- Columns: year, month, total_revenue, total_payments, total_bookings, total_booking_value, avg_stay_days, new_users, active_hosts
- Use this for cross-domain questions (e.g., "How does revenue compare to bookings?").

## fact_payments (transaction-level detail)
- Use this for drill-down: individual transactions, specific booking payments, or filtered transaction lists.
- Columns: payment_id, booking_id, user_id, property_id, host_id, amount, payment_method, status, is_successful, payment_date, payment_month, payment_year, check_in, check_out, booking_total_amount
- is_successful = TRUE when status = 'completed'

## Formatting
- Revenue and amounts should be displayed as USD currency (e.g., $1,234.56).
- Success rate should be displayed as a percentage (e.g., 91.37%).
- Always include the time period in the answer context.
```

---

In [0]:
# Step 4: Add Example SQL Queries (CRITICAL)

## Instructions (follow in Databricks UI)

1. Click the **Configure** tab
2. Click **Examples** in the left panel
3. For each query below:
   - Click **Add example query**
   - **Title** = the natural language question (use exact wording users would type)
   - **SQL** = the query text (copy-paste from below)
   - Click **Save**
4. Add at least 5 queries (Databricks recommends 5+). We'll add 10.

### Why this step matters
> Example SQL queries teach Genie the correct MEASURE() syntax for metric views, the right table names, and the business logic. Genie either uses the example directly or learns from it to answer similar questions.

---

## Example Query 1: What is the total revenue by month?

```sql
SELECT 
  payment_month,
  MEASURE(total_revenue) AS total_revenue,
  MEASURE(total_payments) AS total_payments
FROM workspace.wanderbricks_semantic.mv_revenue_metrics
GROUP BY payment_month
ORDER BY payment_month DESC
```

---

## Example Query 2: Which payment method has the highest failure rate?

```sql
SELECT 
  payment_method,
  MEASURE(failed_payments) AS failed_payments,
  MEASURE(total_payments) AS total_payments,
  (1.0 - MEASURE(payment_success_rate)) * 100 AS failure_rate_pct
FROM workspace.wanderbricks_semantic.mv_revenue_metrics
GROUP BY payment_method
ORDER BY failure_rate_pct DESC
```

---

## Example Query 3: Show me revenue by payment method for 2025

```sql
SELECT 
  payment_method,
  MEASURE(total_revenue) AS revenue,
  MEASURE(total_payments) AS payment_count,
  MEASURE(avg_payment_amount) AS avg_amount
FROM workspace.wanderbricks_semantic.mv_revenue_metrics
WHERE payment_year = 2025
GROUP BY payment_method
ORDER BY revenue DESC
```

---

## Example Query 4: What is the payment success rate trend by month?

```sql
SELECT 
  payment_month,
  MEASURE(payment_success_rate) AS success_rate,
  MEASURE(successful_payments) AS successful,
  MEASURE(failed_payments) AS failed
FROM workspace.wanderbricks_semantic.mv_revenue_metrics
GROUP BY payment_month
ORDER BY payment_month DESC
```

---

## Example Query 5: What is the total revenue and payment count for each year?

```sql
SELECT 
  payment_year,
  MEASURE(total_revenue) AS total_revenue,
  MEASURE(total_payments) AS total_payments,
  MEASURE(avg_payment_amount) AS avg_payment_amount,
  MEASURE(payment_success_rate) AS success_rate
FROM workspace.wanderbricks_semantic.mv_revenue_metrics
GROUP BY payment_year
ORDER BY payment_year DESC
```

---

## Example Query 6: Show me the top 10 highest value completed transactions

```sql
SELECT 
  p.payment_id,
  p.booking_id,
  p.amount,
  p.payment_method,
  p.status,
  p.payment_date,
  u.name AS user_name,
  pr.title AS property_title
FROM workspace.wanderbricks_gold.fact_payments p
LEFT JOIN workspace.wanderbricks_gold.dim_users u ON p.user_id = u.user_id
LEFT JOIN workspace.wanderbricks_gold.dim_properties pr ON p.property_id = pr.property_id
WHERE p.status = 'completed'
ORDER BY p.amount DESC
LIMIT 10
```

---

## Example Query 7: How does revenue compare to bookings by month?

```sql
SELECT 
  year,
  month,
  total_revenue,
  total_payments,
  total_bookings,
  total_booking_value,
  ROUND(total_revenue / NULLIF(total_bookings, 0), 2) AS revenue_per_booking
FROM workspace.wanderbricks_semantic.vw_executive_dashboard
ORDER BY year DESC, month DESC
```

---

## Example Query 8: Which payment methods are growing vs declining year over year?

```sql
SELECT 
  a.payment_method,
  a.payment_year AS current_year,
  MEASURE(a.total_revenue) AS current_revenue,
  b.payment_year AS prior_year,
  MEASURE(b.total_revenue) AS prior_revenue,
  CASE 
    WHEN MEASURE(b.total_revenue) = 0 THEN NULL
    ELSE ROUND((MEASURE(a.total_revenue) - MEASURE(b.total_revenue)) / MEASURE(b.total_revenue) * 100, 2)
  END AS yoy_growth_pct
FROM workspace.wanderbricks_semantic.mv_revenue_metrics a
LEFT JOIN workspace.wanderbricks_semantic.mv_revenue_metrics b 
  ON a.payment_method = b.payment_method 
  AND b.payment_year = a.payment_year - 1
GROUP BY a.payment_method, a.payment_year, b.payment_year
ORDER BY a.payment_method, a.payment_year DESC
```

---

## Example Query 9: Show me all failed payments with booking and user details

```sql
SELECT 
  p.payment_id,
  p.amount,
  p.payment_method,
  p.payment_date,
  u.name AS user_name,
  u.country AS user_country,
  pr.title AS property_title,
  pr.property_type,
  p.booking_total_amount
FROM workspace.wanderbricks_gold.fact_payments p
LEFT JOIN workspace.wanderbricks_gold.dim_users u ON p.user_id = u.user_id
LEFT JOIN workspace.wanderbricks_gold.dim_properties pr ON p.property_id = pr.property_id
WHERE p.status = 'failed'
ORDER BY p.payment_date DESC
LIMIT 50
```

---

## Example Query 10: What is the average payment amount by payment method and year?

```sql
SELECT 
  payment_year,
  payment_method,
  MEASURE(avg_payment_amount) AS avg_payment_amount,
  MEASURE(total_revenue) AS total_revenue,
  MEASURE(total_payments) AS total_payments
FROM workspace.wanderbricks_semantic.mv_revenue_metrics
GROUP BY payment_year, payment_method
ORDER BY payment_year DESC, total_revenue DESC
```

---

In [0]:
# Step 5: Configure Knowledge Store

> **Note:** There is no separate "Knowledge Store" tab in the current UI.
> Knowledge store items are spread across **Configure → Data** (table descriptions) and
> **Configure → Examples** (joins and SQL expressions). See navigation below.

## Instructions (follow in Databricks UI)

1. Click the **Configure** tab
2. The left panel shows: **Data** | **Examples** | **Instructions**

### 5a: Define Join Relationships

**Navigation: Configure → Examples → Add → Joins**

> **Check first!** Primary and foreign keys defined in your Unity Catalog schema are
> **automatically saved** as join relationships. Many joins may already exist.
> Only add the ones below if they're missing.

Click **Add → Joins** for each relationship below:

**fact_payments joins:**
```
workspace.wanderbricks_gold.fact_payments.user_id = workspace.wanderbricks_gold.dim_users.user_id
workspace.wanderbricks_gold.fact_payments.property_id = workspace.wanderbricks_gold.dim_properties.property_id
workspace.wanderbricks_gold.fact_payments.host_id = workspace.wanderbricks_gold.dim_hosts.host_id
workspace.wanderbricks_gold.fact_payments.booking_id = workspace.wanderbricks_gold.fact_bookings.booking_id
```

**fact_bookings joins:**
```
workspace.wanderbricks_gold.fact_bookings.user_id = workspace.wanderbricks_gold.dim_users.user_id
workspace.wanderbricks_gold.fact_bookings.property_id = workspace.wanderbricks_gold.dim_properties.property_id
workspace.wanderbricks_gold.fact_bookings.host_id = workspace.wanderbricks_gold.dim_hosts.host_id
```

**dim_properties joins:**
```
workspace.wanderbricks_gold.dim_properties.host_id = workspace.wanderbricks_gold.dim_hosts.host_id
```

### 5b: Add Table Descriptions

**Navigation: Configure → Data → click a table name → edit description**

For each table, click the table name and add a clear description if not already present:

**mv_revenue_metrics:**
```
Aggregated revenue and payment KPIs by year, month, and payment method. Contains measures: total_revenue, total_payments, avg_payment_amount, successful_payments, failed_payments, payment_success_rate. This is a METRIC VIEW — use MEASURE() function to query measures.
```

**fact_payments:**
```
Transaction-level payment records for Wanderbricks bookings. Each row is one payment attempt. Links to users, properties, hosts, and bookings via foreign keys. Use for drill-down queries, specific transaction lookups, and detailed payment analysis.
```

**vw_executive_dashboard:**
```
Cross-domain executive summary combining revenue, payments, bookings, users, and hosts by year and month. Standard SQL view — no MEASURE() needed. Use for questions that span multiple business domains.
```

### 5c: Add SQL Expressions

**Navigation: Configure → Examples → Add → (select type: measure, filter, or field)**

Add these custom expressions to help Genie understand business logic:

**Expression: failure_rate**
```sql
(1.0 - MEASURE(payment_success_rate)) * 100
```
Description: `Failure rate as a percentage. Calculated as (1 - success_rate) * 100.`

**Expression: revenue_per_booking**
```sql
total_revenue / NULLIF(total_bookings, 0)
```
Description: `Average revenue generated per booking. Uses vw_executive_dashboard columns.`

---

In [0]:
# Step 6: Test with Benchmark Questions

## Instructions (follow in Databricks UI)

1. Click the **Chat** tab (main Genie interface)
2. Make sure your SQL Warehouse is **running** (start it if stopped)
3. Ask each benchmark question below, one at a time
4. For each answer:
   - Check the **generated SQL** (click "Show query" or expand the response)
   - Verify the **results** make sense
   - Use **confidence voting** (👍 / 👎) to mark correct/incorrect answers
   - If wrong, note the issue and refine instructions or add a new example query

---

## Benchmark Questions (Test These)

### Tier 1: Basic Revenue Questions

| # | Question | Expected Behavior |
|---|---|---|
| 1 | "What is the total revenue by month?" | Groups by payment_month, uses MEASURE(total_revenue) |
| 2 | "What is the total revenue for 2025?" | Filters payment_year = 2025, sums total_revenue |
| 3 | "Which payment method generates the most revenue?" | Groups by payment_method, orders by revenue DESC |
| 4 | "What is the average payment amount?" | Uses MEASURE(avg_payment_amount) |

### Tier 2: Payment Performance Questions

| # | Question | Expected Behavior |
|---|---|---|
| 5 | "What is the payment success rate by payment method?" | Groups by payment_method, uses MEASURE(payment_success_rate) |
| 6 | "Which payment method has the most failed payments?" | Groups by payment_method, uses MEASURE(failed_payments), orders DESC |
| 7 | "How has the payment success rate changed over time?" | Groups by payment_month, orders chronologically |

### Tier 3: Cross-Domain Questions

| # | Question | Expected Behavior |
|---|---|---|
| 8 | "How does revenue compare to bookings by month?" | Uses vw_executive_dashboard, shows total_revenue + total_bookings |
| 9 | "What is the revenue per booking by month?" | Uses vw_executive_dashboard, computes revenue/bookings |

### Tier 4: Drill-Down Questions

| # | Question | Expected Behavior |
|---|---|---|
| 10 | "Show me the top 10 highest value transactions" | Uses fact_payments, joins dim_users + dim_properties, orders by amount DESC |
| 11 | "Show me all failed payments with user names" | Uses fact_payments, joins dim_users, filters status = 'failed' |
| 12 | "What is the year-over-year revenue growth by payment method?" | Self-join mv_revenue_metrics on payment_year, computes YoY % |

---

## Evaluation Rubric

For each benchmark question, score:

| Score | Meaning |
|---|---|
| ✅ Correct | SQL runs, results match expected logic, answer is well-formatted |
| ⚠️ Partial | SQL runs but logic is slightly off (e.g., wrong filter, missing column) |
| ❌ Failed | SQL errors, wrong table used, or MEASURE() syntax missing |

**If any question scores ❌:**
1. Copy the generated SQL that Genie produced
2. Identify the issue (wrong syntax, wrong table, wrong filter)
3. Add a corrected example query in Step 4 that covers this pattern
4. Re-test the question

---

In [0]:
# Step 7: Quality Tuning (Ongoing)

## Confidence Voting

After each answer in the Chat tab, you'll see 👍 and 👎 buttons:
- **👍 Correct answer** — reinforces the behavior
- **👎 Wrong answer** — prompts you to provide feedback

Vote on every benchmark question. Databricks uses this feedback to improve response quality.

---

## Common Issues & Fixes

| Issue | Cause | Fix |
|---|---|---|
| Genie doesn't use MEASURE() syntax | Metric view not recognized as metric view | Add more example queries with MEASURE() in Step 4. Ensure table description mentions "METRIC VIEW" |
| Genie queries wrong table | Too many tables attached, or similar table names | Remove unnecessary tables. Add table descriptions that clarify when to use each |
| Revenue shows as raw number, not currency | Missing format instruction | Add to General Instructions: "Display revenue as USD currency ($X,XXX.XX)" |
| Genie can't answer "recent" or "last month" | Doesn't know the latest date in data | Add instruction: "The latest data is from 2025. Use payment_year = 2025 for recent queries." |
| Join fails or returns NULL | Missing join relationship in knowledge store | Add the join relationship in Step 5 (Knowledge store → Join relationships) |
| SQL syntax error on metric views | Missing GROUP BY for dimensions | Add example queries that show GROUP BY with all selected dimensions |

---

## Adding Benchmarks (Optional but Recommended)

1. Click the **Benchmarks** tab
2. Click **Add benchmark**
3. Add the 12 benchmark questions from Step 6
4. Run benchmarks periodically to check accuracy
5. Genie scores itself against your voted answers

---

## Sharing the Agent

When ready to share:
1. Click **Share** in the top right
2. Add users or groups by email
3. Set permission level:
   - **Can view** — user can chat with the agent
   - **Can edit** — user can modify instructions, examples, and data sources
4. Shared users can only see data they have Unity Catalog permissions for

---

## Next Steps After This Agent

Once this agent is working well:

1. **App #2: Booking Analytics Explorer** — New Genie Agent using `mv_booking_metrics`
2. **App #3: Host Performance Coach** — Agent Bricks custom agent using `mv_host_performance`
3. **App #4: Executive Insights Agent** — Agent Bricks + Genie One tool across all 4 metric views
4. **App #5: Guest Analytics Segmentation** — Agent Bricks using `mv_user_analytics`

---

In [0]:
%sql
-- ============================================================
-- REFERENCE: Column Dictionary for All Attached Tables
-- Run this cell to see all columns available to your Genie Agent
-- ============================================================

-- mv_revenue_metrics (METRIC VIEW — use MEASURE())
SELECT 'mv_revenue_metrics' AS source,
       column_name,
       data_type,
       comment
FROM system.information_schema.columns
WHERE table_catalog = 'workspace'
  AND table_schema = 'wanderbricks_semantic'
  AND table_name = 'mv_revenue_metrics'
ORDER BY ordinal_position;

-- vw_executive_dashboard (standard VIEW — no MEASURE() needed)
SELECT 'vw_executive_dashboard' AS source,
       column_name,
       data_type,
       comment
FROM system.information_schema.columns
WHERE table_catalog = 'workspace'
  AND table_schema = 'wanderbricks_semantic'
  AND table_name = 'vw_executive_dashboard'
ORDER BY ordinal_position;

-- fact_payments (transaction-level detail)
SELECT 'fact_payments' AS source,
       column_name,
       data_type,
       comment
FROM system.information_schema.columns
WHERE table_catalog = 'workspace'
  AND table_schema = 'wanderbricks_gold'
  AND table_name = 'fact_payments'
ORDER BY ordinal_position;

-- fact_bookings (booking context)
SELECT 'fact_bookings' AS source,
       column_name,
       data_type,
       comment
FROM system.information_schema.columns
WHERE table_catalog = 'workspace'
  AND table_schema = 'wanderbricks_gold'
  AND table_name = 'fact_bookings'
ORDER BY ordinal_position;

-- dim_users, dim_hosts, dim_properties, dim_date
SELECT 'dim_users' AS source, column_name, data_type, comment
FROM system.information_schema.columns
WHERE table_catalog = 'workspace' AND table_schema = 'wanderbricks_gold' AND table_name = 'dim_users'
UNION ALL
SELECT 'dim_hosts' AS source, column_name, data_type, comment
FROM system.information_schema.columns
WHERE table_catalog = 'workspace' AND table_schema = 'wanderbricks_gold' AND table_name = 'dim_hosts'
UNION ALL
SELECT 'dim_properties' AS source, column_name, data_type, comment
FROM system.information_schema.columns
WHERE table_catalog = 'workspace' AND table_schema = 'wanderbricks_gold' AND table_name = 'dim_properties'
UNION ALL
SELECT 'dim_date' AS source, column_name, data_type, comment
FROM system.information_schema.columns
WHERE table_catalog = 'workspace' AND table_schema = 'wanderbricks_gold' AND table_name = 'dim_date'
ORDER BY source, column_name;